# HARMONY: Interactive Pre-Cluster Validation

This interactive notebook provides an optional **Pre-Cluster Validation** environment for HARMONY (`namespace harmony`, `include/harmony/` + `src/`) on Google Colab.

## Purpose and Scope
Pre-Cluster Validation is an optional verification workflow in a single-node virtualized GPU environment to validate:
- C++17 toolchain compilation via `./build.sh` (`mpicxx`)
- OpenCL runtime discovery on real hardware (e.g. NVIDIA Tesla T4)
- MPI halo exchange across ranks sharing a GPU (`tests/bin/test_async_halo_opencl`)
- Numerical convergence and stability of `bin/kneighbor` and `bin/leanmd` balance kernels

> **Operational Boundary**: Pre-Cluster Validation verifies functional correctness and prevents trivial job failures before PBS batch submission on the HPC cluster. It does not replace multi-node cluster benchmark performance evaluations or support cluster scalability or speedup claims (see ADR-0001 and ADR-0002).

## GPU Runtime Setup
Before running the cells below, enable the GPU accelerator:
1. Navigate to **Runtime** -> **Change runtime type** in the top menu.
2. Set **Hardware accelerator** to **T4 GPU** (or available GPU).
3. Click **Save**.


In [ ]:
# Step 1: Preflight Verification - Check GPU and OpenCL environment
!nvidia-smi
!clinfo -l || true


## Repository Ingestion

Clone the HARMONY repository from GitHub or check out a specific branch, tag, or commit hash for validation.

In [ ]:
# Step 2: Clone or update HARMONY repository
import os

REPO_URL = "https://github.com/lispz1610/HARMONY.git"
BRANCH_OR_TAG = "feature/migrate-to-upstream-harmony"
COMMIT_SHA = ""  # Optional: specify commit SHA to checkout

%cd /content
if os.path.exists("HARMONY"):
    !rm -rf HARMONY
!git clone -b {BRANCH_OR_TAG} {REPO_URL} HARMONY
%cd /content/HARMONY

if COMMIT_SHA:
    !git checkout {COMMIT_SHA}

!git log -n 1 --oneline


## Environment Bootstrap

Install system prerequisites (Open MPI, OpenCL headers, OpenCL ICD loader, and development tools) using `colab/bootstrap.sh`.

In [ ]:
# Step 3: Run environment bootstrap
%cd /content/HARMONY
!bash colab/bootstrap.sh


## C++17 Library, Test, and Benchmark Build

Compile `lib/libharmony.a`, tools (`bin/topo_probe`, `bin/migration_profiler`), benchmark binaries (`bin/kneighbor`, `bin/leanmd`), and test executables (`tests/bin/test_*`) via `colab/build.sh` (delegating to `./build.sh lib tools benchmarks tests`).

In [ ]:
# Step 4: Compile library, tools, benchmarks, and tests
%cd /content/HARMONY
!bash colab/build.sh


## Pre-Cluster Validation Suite Execution

Execute the complete Pre-Cluster Validation runner (`colab/run_validation.sh`). This performs:
1. **Phase 1**: Unit tests (`tests/bin/test_topo_metrics`, `tests/bin/test_algorithms`, `tests/bin/test_numa_cost`, `tests/bin/test_contention_power`, `tests/bin/test_hierarchical`).
2. **Phase 2**: OpenCL asynchronous halo exchange test (`tests/bin/test_async_halo_opencl`) with 1 rank and 2 ranks.
3. **Phase 3**: Fast functional validation runs for `bin/kneighbor` and `bin/leanmd` (`erad` and `hwtopolb`) with finite value checks.

A JSON execution summary is saved to `colab_status.json` in the run directory.

In [ ]:
# Step 5: Execute Pre-Cluster Validation runner
%cd /content/HARMONY
!bash colab/run_validation.sh


## Extended Validation Run (Optional)

Optionally run `./bin/kneighbor` and `./bin/leanmd` with extended iteration counts (`--iterations`) and custom problem sizes (`--n`) to confirm numerical convergence on the virtualized GPU.

In [ ]:
# Step 6: (Optional) Extended validation run with configurable iterations
%cd /content/HARMONY

ITERATIONS = 50
INTERVAL = 5
N = 50000

# Validate kneighbor with 2 ranks
!mpirun --allow-run-as-root --oversubscribe -np 2 \
    ./bin/kneighbor --n {N} --iterations {ITERATIONS} --balance-interval {INTERVAL} --balance-algo erad
!mpirun --allow-run-as-root --oversubscribe -np 2 \
    ./bin/kneighbor --n {N} --iterations {ITERATIONS} --balance-interval {INTERVAL} --balance-algo hwtopolb

# Validate leanmd with 2 ranks
!mpirun --allow-run-as-root --oversubscribe -np 2 \
    ./bin/leanmd --n {N} --iterations {ITERATIONS} --balance-interval {INTERVAL} --balance-algo erad
!mpirun --allow-run-as-root --oversubscribe -np 2 \
    ./bin/leanmd --n {N} --iterations {ITERATIONS} --balance-interval {INTERVAL} --balance-algo hwtopolb
